
# FRE-GT-9743 Assignment 5 Part I: Implicit Function Differentiation

**Setup.** A parametric model $M(X^I)$ has an input (state) vector $X^I$. A vector of atomic functions $V$ is built from the model, with $V_i(X^I)$ meant to reproduce the market quote $X^M_i$, and $\dim(X^I) = \dim(X^M)$, so the system is square:
$$
V(X^I) = X^M .
$$



## Q1. General recipe for solving $X^I$ from $V(X^I) = X^M$

Write the residual
$$
g(X^I) = V(X^I) - X^M = 0 ,
$$
and solve it with a multi-dimensional Newton–Raphson iteration. Let $J(X^I) = \dfrac{\partial V}{\partial X^I}$, with entries $J_{ij} = \dfrac{\partial V_i}{\partial X^I_j}$.

1. Choose an initial guess $X^I_{(0)}$ (for example flat or from the previous calibration).
2. Compute $V(X^I_{(n)})$ and the Jacobian $J(X^I_{(n)})$.
3. Solve the linear system $J\,\Delta = -\big(V(X^I_{(n)}) - X^M\big)$ and update $X^I_{(n+1)} = X^I_{(n)} + \Delta$.
4. Stop when $\|V(X^I_{(n)}) - X^M\|$ is below a tolerance, and optionally when $\|\Delta\|$ is also small.

Practical points:

- The Jacobian is computed analytically where possible (it is needed again in Q2) or by bump-and-reval.
- Newton converges quadratically near the solution but can fail from a poor guess, so a damped step or a line search is used when needed.
- The solution is unique and the iteration is well posed when $J$ is invertible near the root, for example when each $V_i$ is monotonic in its own parameter and the coupling between components is weak.
- If the system were not square, the same loop becomes a least-squares problem (Gauss–Newton or Levenberg–Marquardt).



## Q2. How to compute $\dfrac{dX^I}{dX^M}$ at the optimal $X^I$

At the solution, $X^I$ is an implicit function of $X^M$ defined by
$$
V\big(X^I(X^M)\big) = X^M .
$$
Differentiate both sides with respect to $X^M$ using the chain rule:
$$
\frac{\partial V}{\partial X^I}\,\frac{dX^I}{dX^M} = I
\quad\Longrightarrow\quad
\boxed{\frac{dX^I}{dX^M} = J^{-1}\Big|_{X^I = X^{I*}}}
$$
This is the implicit function theorem. It requires $J$ to be invertible at the solution.

- The Jacobian is the same one Newton used in the last iteration, so the sensitivity is nearly free.
- In practice one solves linear systems with $J$ rather than forming the inverse explicitly.
- For any quantity $\Pi(X^I)$ priced off the calibrated model (for example a trade price), the risk to the market quotes follows from the chain rule:
$$
\frac{d\Pi}{dX^M} = \frac{\partial \Pi}{\partial X^I}\,J^{-1} .
$$
So the model's own sensitivities $\partial\Pi/\partial X^I$ are converted into sensitivities to the instruments traders actually quote, without re-solving the calibration for every bump.



## Q3. Application: calibrating an implied-vol interpolator to three option prices

**Inputs.**

- Three undiscounted European OTM option prices $P_1, P_2, P_3$ at strikes $K_1 < K_{atm} < K_3$. Here $P_1$ is an OTM put, $P_2$ is the ATM option, and $P_3$ is an OTM call.
- A vol interpolator $\sigma(K;\theta)$ with parameter vector $\theta \in \mathbb{R}^3$ (three parameters for three quotes, so the system is square).
- Forward $F$ and expiry $T$.

**Mapping to Part I.**

- Market quotes: $X^M = (P_1, P_2, P_3)$.
- Model state: $X^I = \theta$.
- Atomic functions: the undiscounted Black price at each strike, using the interpolated vol,
$$
V_i(\theta) = \text{Black}\big(F, K_i, \sigma(K_i;\theta), T\big).
$$

**Finding the unique $\theta$.** Solve $V(\theta) = P$ with the Newton recipe from Q1. The Jacobian factors into two pieces:
$$
J_{ij} = \frac{\partial V_i}{\partial \theta_j}
= \underbrace{\frac{\partial \text{Black}_i}{\partial \sigma}}_{\text{vega}_i}\;
\underbrace{\frac{\partial \sigma(K_i;\theta)}{\partial \theta_j}}_{S_{ij}}
\quad\Longleftrightarrow\quad
J = \text{diag}(\text{vega})\,S .
$$

- $S_{ij}$ is the gradient of the interpolator with respect to its parameters. (This is the same kind of object as `gradient_wrt_ordinate` in the interpolator from Assignment 1.)
- Uniqueness: the Black price is strictly increasing in vol, so $\text{vega}_i > 0$ and each option corresponds to exactly one implied vol. If the interpolator maps $\theta$ to the three vols $\big(\sigma(K_1),\sigma(K_2),\sigma(K_3)\big)$ in a one-to-one way (so $S$ is invertible), then $J$ is invertible and the $\theta$ matching the market is unique.
- Equivalent two-step view: first invert Black at each strike to get implied vols $\sigma_i^{mkt}$ (a 1-D root-find per option), then solve $\sigma(K_i;\theta) = \sigma_i^{mkt}$ for $\theta$.

**How $\sigma(K)$ changes with $\theta$.** For any strike $K$, the sensitivity to the parameters is the interpolator gradient,
$$
\frac{\partial \sigma(K;\theta)}{\partial \theta} .
$$
**How $\sigma(K)$ changes with the market prices.** Combine with Q2:
$$
\frac{d\sigma(K)}{dP}
= \frac{\partial \sigma(K;\theta)}{\partial \theta}\,\frac{d\theta}{dP}
= \frac{\partial \sigma(K;\theta)}{\partial \theta}\,J^{-1}
= \frac{\partial \sigma(K;\theta)}{\partial \theta}\,S^{-1}\,\text{diag}\!\left(\frac{1}{\text{vega}}\right).
$$
This says how the whole vol curve moves when one of the three quoted option prices moves by one unit, which is the bucketed vol risk with respect to the traded instruments.

**What $\theta$ might mean to a trader (intuition).** With three quotes, a natural choice is three parameters that traders already think in:

1. **Level:** the ATM vol, which shifts the whole smile up or down.
2. **Skew:** the slope of the smile, for example the risk reversal (call-wing vol minus put-wing vol), which tilts the curve.
3. **Curvature:** the smile or butterfly (average wing vol minus ATM vol), which raises or lowers both wings together.

The simplest alternative is $\theta$ equal to the node vols at the three strikes (put wing, ATM, call wing). Under that choice, $\partial\sigma(K)/\partial\theta$ shows directly how much of each wing and the ATM node feeds into any other strike. With the level, skew and curvature choice, the sensitivities map onto the vega, risk-reversal and butterfly exposures that desks already quote and hedge.
